# Challenge W4: NLP - Fake News
## Steps:
    1. Import data
    2. EDA
    3. Data Splitting
    4. Text Preprocessing 
    5. Feature Extraction
    6. Model Training

1. Libraries:

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
#import seaborn as sns

import re
import string

from sklearn.model_selection import train_test_split

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

2. Load data

In [3]:
df = pd.read_csv("./dataset/data.csv")

In [4]:
df.shape
df.head()       
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 39942 entries, 0 to 39941
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    39942 non-null  int64
 1   title    39942 non-null  str  
 2   text     39942 non-null  str  
 3   subject  39942 non-null  str  
 4   date     39942 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.5 MB


3. EDA

In [5]:
df.isnull().sum() # missing values?

label      0
title      0
text       0
subject    0
date       0
dtype: int64

In [6]:
print("Duplicate rows:", df.duplicated().sum()) # Duplicates?


Duplicate rows: 201


In [7]:
df.sample(5, random_state=42)

,label,title,text,subject,date
6524,1,Oil business seen in strong position as Trump ...,(This January 3 story was corrected to remove...,politicsNews,"January 3, 2017"
30902,0,WHOA! COLLEGE SNOWFLAKE FREAKS OUT: Screams Fo...,So much for healthy debate on college campus I...,politics,"May 12, 2017"
36459,0,CRONY CORRUPT POLITICS: Obama Admin BLOCKED FB...,The information is spilling out little by litt...,Government News,"Aug 11, 2016"
9801,1,Cruz campaign vetting Fiorina as a possible VP...,WASHINGTON (Reuters) - U.S. Republican preside...,politicsNews,"April 25, 2016"
25638,0,Minnesota Woman Writes Amazing F*ck Off Lette...,"Attention, conservative men. This one is for y...",News,"July 2, 2016"


In [8]:
print("Unique subjects:", df['subject'].nunique())
df['subject'].value_counts()

Unique subjects: 6


subject
politicsNews       11272
News                9050
worldnews           8727
politics            6841
left-news           2482
Government News     1570
Name: count, dtype: int64

In [10]:
pd.crosstab(df['subject'], df['label']) # Because subject and label are correlated we should drop "subject" column to avoid data leakage.

label,0,1
subject,,
Government News,1570,0
News,9050,0
left-news,2482,0
politics,6841,0
politicsNews,0,11272
worldnews,0,8727


## TRAIN TEST VALIDATION SPLIT

In [11]:
from sklearn.model_selection import train_test_split

# Dropping columns subject and date as the have been shown to not carry important information
df = df.drop(columns=['subject','date']) 


# Split into features and target
# Adjust the target column name to match your dataset (e.g. 'label', 'class', 'Spam/Ham')
X = df.drop(columns=['label'])   # <-- replace 'label' with your actual target column
y = df['label']

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: Split the remaining 30% into 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display dataset shapes
print(f"Train shape:      {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape:       {X_test.shape}")

# Display class balance
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))

print("\nValidation class balance:")
print(y_val.value_counts(normalize=True))

print("\nTest class balance:")
print(y_test.value_counts(normalize=True))

Train shape:      (27959, 2)
Validation shape: (5991, 2)
Test shape:       (5992, 2)

Train class balance:
label
1    0.500697
0    0.499303
Name: proportion, dtype: float64

Validation class balance:
label
1    0.500751
0    0.499249
Name: proportion, dtype: float64

Test class balance:
label
1    0.500668
0    0.499332
Name: proportion, dtype: float64
